In [1]:
%pip install pymongo mysql-connector-python

import mysql.connector
from pymongo import MongoClient
import json
import os
import datetime


mysql_conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="saludusm"
)
mysql_cursor = mysql_conn.cursor(dictionary=True)

uri = "mongodb://192.168.1.8:30001,192.168.1.8:30002,192.168.1.8:30003/?replicaSet=rs0"
mongo_client = MongoClient(uri, serverSelectionTimeoutMS=5000)
db_mongo = mongo_client["salud_usm"]

try:
    print("Primario:", mongo_client.primary)
    print("Nodos:", list(mongo_client.nodes))
    print("Conexión a MongoDB establecida")
except Exception as e:
    print(f"Error: {e}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Primario: None
Nodos: []
Conexión a MongoDB establecida


In [2]:
import datetime

def convertir_fechas(doc):
    if isinstance(doc, dict):
        return {k: convertir_fechas(v) for k, v in doc.items()}
    elif isinstance(doc, list):
        return [convertir_fechas(v) for v in doc]
    elif isinstance(doc, (datetime.date, datetime.datetime)):
        return doc.isoformat()
    else:
        return doc

catalogos = ['prevision', 'especialidad', 'estado_cita', 'centro_medico']
for tabla in catalogos:
    mysql_cursor.execute(f"SELECT * FROM {tabla}")
    datos = mysql_cursor.fetchall()
    if datos:
        db_mongo[tabla].drop()
        datos_limpios = [convertir_fechas(d) for d in datos]
        db_mongo[tabla].insert_many(datos_limpios)
        print(f"Migrados {len(datos)} registros a '{tabla}'.")


mysql_cursor.execute("""
    SELECT p.id_paciente, u.rut, u.nombre, u.apellido, u.email, 
           u.telefono, u.sexo, p.id_prevision
    FROM paciente p 
    JOIN usuario u ON p.email = u.email
""")
pacientes = mysql_cursor.fetchall()
if pacientes:
    db_mongo.paciente.drop()
    pacientes_limpios = [convertir_fechas(p) for p in pacientes]
    db_mongo.paciente.insert_many(pacientes_limpios)
    print(f"Migrados {len(pacientes)} pacientes.")

mysql_cursor.execute("""
    SELECT m.id_medico, u.rut, u.nombre, u.apellido, u.email, 
           u.telefono, u.sexo
    FROM medico m 
    JOIN usuario u ON m.email = u.email
""")
medicos = mysql_cursor.fetchall()

for m in medicos:
    mysql_cursor.execute(
        f"SELECT id_especialidad FROM medico_especialidad WHERE id_medico = {m['id_medico']}"
    )
    m['id_especialidades'] = [e['id_especialidad'] for e in mysql_cursor.fetchall()]

if medicos:
    db_mongo.medico.drop()
    medicos_limpios = [convertir_fechas(m) for m in medicos]
    db_mongo.medico.insert_many(medicos_limpios)
    print(f"Migrados {len(medicos)} médicos.")


mysql_cursor.execute("SELECT * FROM cita")
citas = mysql_cursor.fetchall()

for c in citas:
    mysql_cursor.execute(
        f"SELECT id_atencion, motivo, observaciones FROM atencion WHERE id_cita = {c['id_cita']}"
    )
    atencion = mysql_cursor.fetchone()
    if atencion:
        mysql_cursor.execute(f"""
            SELECT d.codigo_cie10, d.descripcion 
            FROM diagnostico_atencion da 
            JOIN diagnostico d ON da.id_diagnostico = d.id_diagnostico 
            WHERE da.id_atencion = {atencion['id_atencion']}
        """)
        atencion['diagnosticos'] = mysql_cursor.fetchall()
        
        mysql_cursor.execute(f"""
            SELECT m.nombre as medicamento, r.dosis, r.dias_tratamiento 
            FROM receta_linea r 
            JOIN medicamento m ON r.id_medicamento = m.id_medicamento 
            WHERE r.id_atencion = {atencion['id_atencion']}
        """)
        atencion['recetas'] = mysql_cursor.fetchall()
        
        c['atencion'] = atencion

if citas:
    db_mongo.cita.drop()
    citas_limpias = [convertir_fechas(c) for c in citas]
    db_mongo.cita.insert_many(citas_limpias)
    print(f"Migradas {len(citas)} citas con atenciones anidadas.")

Migrados 4 registros a 'prevision'.
Migrados 8 registros a 'especialidad'.
Migrados 5 registros a 'estado_cita'.
Migrados 4 registros a 'centro_medico'.
Migrados 7 pacientes.
Migrados 4 médicos.
Migradas 21 citas con atenciones anidadas.


In [4]:
from datetime import datetime

# Parámetros de ejemplo
rut_medico = "12345678-9"  # Reemplaza con un RUT real de tu BD
fecha = "2026-11-15"       # Formato YYYY-MM-DD

# Convertir fecha a rango de día
fecha_inicio = datetime.strptime(fecha, "%Y-%m-%d")
fecha_fin = fecha_inicio.replace(hour=23, minute=59, second=59)

pipeline = [
    # 1. Filtrar por médico y fecha
    {
        "$match": {
            "rut_medico": rut_medico,
            "fecha_hora": {"$gte": fecha_inicio, "$lte": fecha_fin}
        }
    },
    # 2. Unir con pacientes para obtener nombre y previsión
    {
        "$lookup": {
            "from": "paciente",
            "localField": "id_paciente",
            "foreignField": "id_paciente",
            "as": "paciente_info"
        }
    },
    {"$unwind": "$paciente_info"},
    # 3. Unir con prevision
    {
        "$lookup": {
            "from": "prevision",
            "localField": "paciente_info.id_prevision",
            "foreignField": "id_prevision",
            "as": "prevision_info"
        }
    },
    {"$unwind": "$prevision_info"},
    # 4. Unir con centro
    {
        "$lookup": {
            "from": "centro_medico",
            "localField": "id_centro",
            "foreignField": "id_centro",
            "as": "centro_info"
        }
    },
    {"$unwind": "$centro_info"},
    # 5. Unir con especialidad
    {
        "$lookup": {
            "from": "especialidad",
            "localField": "id_especialidad",
            "foreignField": "id_especialidad",
            "as": "especialidad_info"
        }
    },
    {"$unwind": "$especialidad_info"},
    # 6. Ordenar por hora
    {"$sort": {"fecha_hora": 1}},
    # 7. Proyectar campos
    {
        "$project": {
            "_id": 0,
            "hora": {"$dateToString": {"format": "%H:%M", "date": "$fecha_hora"}},
            "paciente": {"$concat": ["$paciente_info.nombre", " ", "$paciente_info.apellido"]},
            "prevision": "$prevision_info.nombre",
            "centro": "$centro_info.nombre",
            "especialidad": "$especialidad_info.nombre",
            "estado": "$estado"
        }
    }
]

resultados = list(db_mongo.cita.aggregate(pipeline))
for r in resultados:
    print(r)

In [ ]:
SELECT 
    DATE_FORMAT(c.fecha_hora, '%H:%i') AS hora,
    CONCAT(p.nombre, ' ', p.apellido) AS paciente,
    pv.nombre AS prevision,
    cm.nombre AS centro,
    e.nombre AS especialidad,
    ec.nombre AS estado
FROM cita c
JOIN paciente pa ON c.id_paciente = pa.id_paciente
JOIN usuario p ON pa.email = p.email
JOIN prevision pv ON pa.id_prevision = pv.id_prevision
JOIN centro_medico cm ON c.id_centro = cm.id_centro
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
JOIN estado_cita ec ON c.id_estado = ec.id_estado
WHERE c.id_medico = (SELECT id_medico FROM medico WHERE rut = '12345678-9')
  AND DATE(c.fecha_hora) = '2026-11-15'
ORDER BY c.fecha_hora;

In [5]:
rut_paciente = "98765432-1"  # Reemplaza con un RUT real

pipeline = [
    # 1. Filtrar por paciente
    {"$match": {"rut_paciente": rut_paciente}},
    # 2. Unir con médico
    {
        "$lookup": {
            "from": "medico",
            "localField": "id_medico",
            "foreignField": "id_medico",
            "as": "medico_info"
        }
    },
    {"$unwind": "$medico_info"},
    # 3. Unir con centro
    {
        "$lookup": {
            "from": "centro_medico",
            "localField": "id_centro",
            "foreignField": "id_centro",
            "as": "centro_info"
        }
    },
    {"$unwind": "$centro_info"},
    # 4. Unir con especialidad
    {
        "$lookup": {
            "from": "especialidad",
            "localField": "id_especialidad",
            "foreignField": "id_especialidad",
            "as": "especialidad_info"
        }
    },
    {"$unwind": "$especialidad_info"},
    # 5. Ordenar por fecha descendente
    {"$sort": {"fecha_hora": -1}},
    # 6. Proyectar
    {
        "$project": {
            "_id": 0,
            "fecha": "$fecha_hora",
            "centro": "$centro_info.nombre",
            "especialidad": "$especialidad_info.nombre",
            "medico": {"$concat": ["$medico_info.nombre", " ", "$medico_info.apellido"]},
            "motivo": "$atencion.motivo",
            "diagnosticos": "$atencion.diagnosticos",
            "recetas": "$atencion.recetas"
        }
    }
]

resultados = list(db_mongo.cita.aggregate(pipeline))
for r in resultados:
    print(r)

In [ ]:
SELECT 
    c.fecha_hora AS fecha,
    cm.nombre AS centro,
    e.nombre AS especialidad,
    CONCAT(m.nombre, ' ', m.apellido) AS medico,
    a.motivo,
    GROUP_CONCAT(CONCAT(d.codigo_cie10, ' - ', d.descripcion) SEPARATOR '; ') AS diagnosticos,
    GROUP_CONCAT(CONCAT(r.medicamento, ' - ', r.dosis, ' - ', r.dias_tratamiento, 'd') SEPARATOR '; ') AS recetas
FROM cita c
JOIN paciente pa ON c.id_paciente = pa.id_paciente
JOIN usuario up ON pa.email = up.email
JOIN medico me ON c.id_medico = me.id_medico
JOIN usuario um ON me.email = um.email
JOIN centro_medico cm ON c.id_centro = cm.id_centro
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
JOIN atencion a ON a.id_cita = c.id_cita
LEFT JOIN diagnostico_atencion da ON da.id_atencion = a.id_atencion
LEFT JOIN diagnostico d ON da.id_diagnostico = d.id_diagnostico
LEFT JOIN receta_linea r ON r.id_atencion = a.id_atencion
WHERE up.rut = '98765432-1'
GROUP BY c.id_cita, c.fecha_hora, cm.nombre, e.nombre, um.nombre, um.apellido, a.motivo
ORDER BY c.fecha_hora DESC;

In [6]:
pipeline = [
    # 1. Unir con estado_cita
    {
        "$lookup": {
            "from": "estado_cita",
            "localField": "id_estado",
            "foreignField": "id_estado",
            "as": "estado_info"
        }
    },
    {"$unwind": "$estado_info"},
    # 2. Agrupar por centro
    {
        "$group": {
            "_id": "$id_centro",
            "total_citas": {"$sum": 1},
            "no_asistio": {
                "$sum": {"$cond": [{"$eq": ["$estado_info.nombre", "No Asistio"]}, 1, 0]}
            }
        }
    },
    # 3. Calcular porcentaje
    {
        "$project": {
            "total_citas": 1,
            "no_asistio": 1,
            "porcentaje": {
                "$round": [
                    {"$multiply": [{"$divide": ["$no_asistio", "$total_citas"]}, 100]},
                    2
                ]
            }
        }
    },
    # 4. Filtrar > 15%
    {"$match": {"porcentaje": {"$gt": 15}}},
    # 5. Ordenar descendente
    {"$sort": {"porcentaje": -1}},
    # 6. Unir con centro para nombre
    {
        "$lookup": {
            "from": "centro_medico",
            "localField": "_id",
            "foreignField": "id_centro",
            "as": "centro_info"
        }
    },
    {"$unwind": "$centro_info"},
    # 7. Proyectar
    {
        "$project": {
            "_id": 0,
            "centro": "$centro_info.nombre",
            "total_citas": 1,
            "no_asistio": 1,
            "porcentaje": 1
        }
    }
]

resultados = list(db_mongo.cita.aggregate(pipeline))
for r in resultados:
    print(r)

{'total_citas': 9, 'no_asistio': 2, 'porcentaje': 22.22, 'centro': 'Centro SaludUSM Casa Central'}


In [ ]:
SELECT 
    cm.nombre AS centro,
    COUNT(c.id_cita) AS total_citas,
    SUM(CASE WHEN ec.nombre = 'No Asistio' THEN 1 ELSE 0 END) AS no_asistio,
    ROUND(
        (SUM(CASE WHEN ec.nombre = 'No Asistio' THEN 1 ELSE 0 END) / COUNT(c.id_cita)) * 100, 
        2
    ) AS porcentaje
FROM centro_medico cm
JOIN cita c ON cm.id_centro = c.id_centro
JOIN estado_cita ec ON c.id_estado = ec.id_estado
GROUP BY cm.id_centro, cm.nombre
HAVING porcentaje > 15
ORDER BY porcentaje DESC;

In [7]:
especialidad_buscada = "Pediatria"  # Reemplaza con una especialidad real

pipeline = [
    # 1. Unir con especialidad
    {
        "$lookup": {
            "from": "especialidad",
            "localField": "id_especialidad",
            "foreignField": "id_especialidad",
            "as": "esp_info"
        }
    },
    {"$unwind": "$esp_info"},
    # 2. Filtrar por especialidad
    {"$match": {"esp_info.nombre": especialidad_buscada}},
    # 3. Desanidar recetas
    {"$unwind": "$atencion.recetas"},
    # 4. Agrupar por medicamento
    {
        "$group": {
            "_id": "$atencion.recetas.medicamento",
            "total": {"$sum": 1}
        }
    },
    # 5. Ordenar descendente
    {"$sort": {"total": -1, "_id": 1}},  # Desempate por nombre alfabético
    # 6. Limitar a 5
    {"$limit": 5},
    # 7. Proyectar
    {
        "$project": {
            "_id": 0,
            "medicamento": "$_id",
            "total": 1
        }
    }
]

resultados = list(db_mongo.cita.aggregate(pipeline))
for r in resultados:
    print(r)

{'total': 3, 'medicamento': 'Paracetamol'}


In [ ]:
SELECT 
    m.nombre AS medicamento,
    COUNT(*) AS total
FROM cita c
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
JOIN atencion a ON a.id_cita = c.id_cita
JOIN receta_linea r ON r.id_atencion = a.id_atencion
JOIN medicamento m ON r.id_medicamento = m.id_medicamento
WHERE e.nombre = 'Pediatria'
GROUP BY m.id_medicamento, m.nombre
ORDER BY total DESC, m.nombre ASC
LIMIT 5;

In [8]:
pipeline = [
    # 1. Unir con estado_cita para filtrar Atendida
    {
        "$lookup": {
            "from": "estado_cita",
            "localField": "id_estado",
            "foreignField": "id_estado",
            "as": "estado_info"
        }
    },
    {"$unwind": "$estado_info"},
    {"$match": {"estado_info.nombre": "Atendida"}},
    # 2. Unir con paciente para obtener nombre
    {
        "$lookup": {
            "from": "paciente",
            "localField": "id_paciente",
            "foreignField": "id_paciente",
            "as": "pac_info"
        }
    },
    {"$unwind": "$pac_info"},
    # 3. Unir con usuario para nombre del paciente
    {
        "$lookup": {
            "from": "usuario",
            "localField": "pac_info.email",
            "foreignField": "email",
            "as": "user_info"
        }
    },
    {"$unwind": "$user_info"},
    # 4. Unir con especialidad para nombre
    {
        "$lookup": {
            "from": "especialidad",
            "localField": "id_especialidad",
            "foreignField": "id_especialidad",
            "as": "esp_info"
        }
    },
    {"$unwind": "$esp_info"},
    # 5. Agrupar por paciente
    {
        "$group": {
            "_id": "$id_paciente",
            "nombre": {"$first": {"$concat": ["$user_info.nombre", " ", "$user_info.apellido"]}},
            "especialidades": {"$addToSet": "$esp_info.nombre"}
        }
    },
    # 6. Filtrar pacientes con 3+ especialidades
    {
        "$match": {
            "$expr": {"$gte": [{"$size": "$especialidades"}, 3]}
        }
    },
    # 7. Proyectar
    {
        "$project": {
            "_id": 0,
            "nombre": 1,
            "cantidad": {"$size": "$especialidades"},
            "especialidades": 1
        }
    },
    # 8. Ordenar por cantidad descendente
    {"$sort": {"cantidad": -1}}
]

resultados = list(db_mongo.cita.aggregate(pipeline))
for r in resultados:
    print(r)

In [ ]:
SELECT 
    CONCAT(u.nombre, ' ', u.apellido) AS paciente,
    COUNT(DISTINCT c.id_especialidad) AS cantidad,
    GROUP_CONCAT(DISTINCT e.nombre SEPARATOR ', ') AS especialidades
FROM cita c
JOIN estado_cita ec ON c.id_estado = ec.id_estado
JOIN paciente p ON c.id_paciente = p.id_paciente
JOIN usuario u ON p.email = u.email
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
WHERE ec.nombre = 'Atendida'
GROUP BY p.id_paciente, u.nombre, u.apellido
HAVING cantidad >= 3
ORDER BY cantidad DESC;

In [9]:
from pymongo import MongoClient

uri = "mongodb://192.168.1.8:30001,192.168.1.8:30002,192.168.1.8:30003/?replicaSet=rs0"
client = MongoClient(uri)

# Obtener el estado del replica set
status = client.admin.command("replSetGetStatus")

print("Estado del Replica Set:", status["set"])
print("Nodos:")
for member in status["members"]:
    print(f"  - {member['name']}: {member['stateStr']} (health: {member['health']})")

print(f"\nPrimario actual: {client.primary}")

Estado del Replica Set: rs0
Nodos:
  - 192.168.1.8:30001: PRIMARY (health: 1.0)
  - 192.168.1.8:30002: SECONDARY (health: 1.0)
  - 192.168.1.8:30003: SECONDARY (health: 1.0)

Primario actual: ('192.168.1.8', 30001)


In [3]:
from pymongo import MongoClient
from pymongo.write_concern import WriteConcern
from datetime import datetime

# Conexión al replica set (para escritura)
uri_rs = "mongodb://192.168.1.8:30001,192.168.1.8:30002,192.168.1.8:30003/?replicaSet=rs0"
client_rs = MongoClient(uri_rs)
db_rs = client_rs["salud_usm"]

# Conexiones directas a cada nodo (para lectura)
uri_nodo1 = "mongodb://192.168.1.8:30001/?directConnection=true"
uri_nodo2 = "mongodb://192.168.1.8:30002/?directConnection=true"
uri_nodo3 = "mongodb://192.168.1.8:30003/?directConnection=true"

client1 = MongoClient(uri_nodo1)
client2 = MongoClient(uri_nodo2)
client3 = MongoClient(uri_nodo3)

db1 = client1["salud_usm"]
db2 = client2["salud_usm"]
db3 = client3["salud_usm"]


print("=" * 60)
print("OPERACIÓN 1: Insertar nueva cita")
print("=" * 60)

# Obtener el id_estado de "Reservada"
estado_reservada = db_rs.estado_cita.find_one({"nombre": "Reservada"})
id_estado_reservada = estado_reservada["id_estado"]

nueva_cita = {
    "id_cita": 9999,  # ID manual para identificarla
    "id_paciente": 1,
    "id_medico": 1,
    "id_especialidad": 1,
    "id_centro": 1,
    "fecha_hora": datetime(2026, 12, 1, 10, 0, 0),
    "id_estado": id_estado_reservada,
    "creada_en": datetime.now()
}

# Insertar con WriteConcern majority
coleccion_cita = db_rs.cita.with_options(write_concern=WriteConcern("majority"))
resultado = coleccion_cita.insert_one(nueva_cita)
print(f"Documento insertado con _id: {resultado.inserted_id}")

# Verificar en cada nodo
print("\nVerificando en cada nodo...")
for i, db_nodo in enumerate([db1, db2, db3], start=1):
    doc = db_nodo.cita.find_one({"id_cita": 9999})
    print(f"  Nodo {i}: {'ENCONTRADO' if doc else 'NO ENCONTRADO'}")
    if doc:
        print(f"    - id_cita: {doc['id_cita']}, estado: {doc['id_estado']}")


print("\n" + "=" * 60)
print("OPERACIÓN 2: Actualizar citas vencidas")
print("=" * 60)

# Obtener el id_estado de "Cancelada"
estado_cancelada = db_rs.estado_cita.find_one({"nombre": "Cancelada"})
id_estado_cancelada = estado_cancelada["id_estado"]

# Actualizar citas vencidas (fecha_hora < ahora y estado Reservada o Confirmada)
fecha_actual = datetime.now()
resultado_update = coleccion_cita.update_many(
    {
        "fecha_hora": {"$lt": fecha_actual},
        "id_estado": {"$in": [id_estado_reservada, 2]}  # Reservada o Confirmada
    },
    {"$set": {"id_estado": id_estado_cancelada}}
)
print(f"Documentos actualizados: {resultado_update.modified_count}")

# Verificar en cada nodo
print("\nVerificando en cada nodo...")
for i, db_nodo in enumerate([db1, db2, db3], start=1):
    count = db_nodo.cita.count_documents({"id_cita": 9999, "id_estado": id_estado_cancelada})
    print(f"  Nodo {i}: {'ACTUALIZADO' if count > 0 else 'NO ACTUALIZADO'}")


print("\n" + "=" * 60)
print("OPERACIÓN 3: Eliminar una línea de receta")
print("=" * 60)

# Primero, asegurémonos de que hay al menos una atención con recetas
atencion_con_recetas = db_rs.cita.find_one(
    {"atencion.recetas": {"$exists": True, "$ne": []}}
)

if atencion_con_recetas:
    id_cita = atencion_con_recetas["id_cita"]
    print(f"Usando cita con id: {id_cita}")
    
    # Obtener el nombre del primer medicamento
    primer_medicamento = atencion_con_recetas["atencion"]["recetas"][0]["medicamento"]
    print(f"Medicamento a eliminar: {primer_medicamento}")
    
    # Eliminar la primera receta del arreglo
    resultado_receta = db_rs.cita.update_one(
        {"id_cita": id_cita},
        {"$pop": {"atencion.recetas": -1}}  # -1 = elimina el primero
    )
    print(f"Receta eliminada: {resultado_receta.modified_count} documento modificado")
    
    # Verificar en cada nodo
    print("\nVerificando en cada nodo...")
    for i, db_nodo in enumerate([db1, db2, db3], start=1):
        doc = db_nodo.cita.find_one({"id_cita": id_cita})
        if doc and "atencion" in doc and "recetas" in doc["atencion"]:
            recetas = doc["atencion"]["recetas"]
            tiene_medicamento = any(r["medicamento"] == primer_medicamento for r in recetas)
            print(f"  Nodo {i}: {'AÚN TIENE' if tiene_medicamento else ' ELIMINADO'} (recetas: {len(recetas)})")
        else:
            print(f"  Nodo {i}: Sin recetas")
else:
    print("⚠️ No se encontró ninguna atención con recetas. Asegúrate de haber migrado los datos.")

OPERACIÓN 1: Insertar nueva cita
Documento insertado con _id: 6ae9319bb3fe0e97edb524f1

Verificando en cada nodo...
  Nodo 1: ENCONTRADO
    - id_cita: 9999, estado: 1
  Nodo 2: ENCONTRADO
    - id_cita: 9999, estado: 1
  Nodo 3: ENCONTRADO
    - id_cita: 9999, estado: 1

OPERACIÓN 2: Actualizar citas vencidas
Documentos actualizados: 0

Verificando en cada nodo...
  Nodo 1: NO ACTUALIZADO
  Nodo 2: NO ACTUALIZADO
  Nodo 3: NO ACTUALIZADO

OPERACIÓN 3: Eliminar una línea de receta
Usando cita con id: 1
Medicamento a eliminar: Losartan
Receta eliminada: 1 documento modificado

Verificando en cada nodo...
  Nodo 1: AÚN TIENE (recetas: 5)
  Nodo 2: AÚN TIENE (recetas: 5)
  Nodo 3: AÚN TIENE (recetas: 5)


In [11]:
import json
import os

carpeta_json = 'JSON'
os.makedirs(carpeta_json, exist_ok=True)

colecciones = db_mongo.list_collection_names()
for col in colecciones:
    datos = list(db_mongo[col].find({}, {'_id': 0}))
    ruta = os.path.join(carpeta_json, f"{col}.json")
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(datos, f, ensure_ascii=False, indent=4, default=str)
    print(f"Exportado {col}.json ({len(datos)} documentos)")

Exportado medico.json (4 documentos)
Exportado prevision.json (4 documentos)
Exportado cita.json (22 documentos)
Exportado especialidad.json (8 documentos)
Exportado paciente.json (7 documentos)
Exportado test.json (2 documentos)
Exportado estado_cita.json (5 documentos)
Exportado centro_medico.json (4 documentos)
